# Task 02：MDP、Bellman 方程与动态规划

本 Notebook 是可执行讲义。算法唯一实现位于 `src/`；这里用小 MDP 和 GridWorld 把公式、数值与图连接起来。

学习目标：理解 MDP、手算 Bellman backup、评估固定策略、实现策略改进，并比较 policy iteration 与 value iteration。

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / "task-02-mdp-dp", Path.cwd().parent]
TASK_DIR = next((p.resolve() for p in candidates if (p / "src" / "mdp.py").is_file()), None)
assert TASK_DIR is not None, "请从仓库根、任务根或 notebooks/ 目录启动 Notebook"
if str(TASK_DIR) not in sys.path:
    sys.path.insert(0, str(TASK_DIR))

import matplotlib.pyplot as plt
import numpy as np
from src.dynamic_programming import (
    bellman_expectation_backup,
    iterative_policy_evaluation,
    policy_iteration,
    value_iteration,
)
from src.gridworld import ACTION_NAMES, GridWorld
from src.mdp import TabularMDP
from src.visualization import plot_value_and_policy, policy_labels

print("Task directory:", TASK_DIR)
print("Actions:", dict(enumerate(ACTION_NAMES)))

## 1. 一个可手算的两状态 MDP

Bellman expectation backup 是两层期望：先对环境转移求和，再按策略动作概率求和。

$$V^{\pi}(s)=\sum_a\pi(a|s)\sum_{s',r}p(s',r|s,a)[r+\gamma(1-d)V(s')]$$

其中 `d=terminated`。下面故意给终止状态一个很大的数组值；正确 target 仍不能 bootstrap 它。

In [ ]:
tiny = TabularMDP([
    [
        [(1.0, 0, 0.0, False)],
        [(1.0, 1, 2.0, True)],
    ],
    [
        [(1.0, 1, 0.0, True)],
        [(1.0, 1, 0.0, True)],
    ],
])
values = np.array([4.0, 999.0])
policy_row = np.array([0.25, 0.75])
backup = bellman_expectation_backup(tiny, 0, values, policy_row, gamma=0.5)
print("Bellman expectation backup:", backup)
print("手算: 0.25*(0+0.5*4) + 0.75*2 =", 0.25 * 2 + 0.75 * 2)

## 2. GridWorld 的转移语义

状态按 row-major 编号，动作顺序是上、右、下、左。非终止状态每步奖励 -1；撞墙留在原地；进入角落后 `terminated=True`。

In [ ]:
env = GridWorld(rows=4, cols=4)
for action, name in enumerate(ACTION_NAMES):
    transition = env.transitions[1][action][0]
    print(f"state=1, action={name:>5} -> next={transition.next_state:2d}, "
          f"reward={transition.reward:4.1f}, terminated={transition.terminated}")

## 3. Policy evaluation

给定均匀随机策略，反复同步应用 Bellman expectation operator，直到 `max_s |V_new(s)-V_old(s)| < theta`。经典 4×4 结果是很有价值的 golden test。

In [ ]:
random_policy = env.uniform_random_policy()
evaluation = iterative_policy_evaluation(
    env, random_policy, gamma=1.0, theta=1e-10, max_iterations=20_000
)
print("converged:", evaluation.converged)
print("sweeps:", evaluation.iterations, "final delta:", evaluation.delta)
print(np.round(evaluation.values.reshape(4, 4), 3))

## 4. Policy iteration 与 value iteration

Policy iteration 交替进行完整评估和贪心改进；value iteration 每个 sweep 直接使用 Bellman optimality backup。二者应得到相同的最优价值与最优动作集合。

In [ ]:
pi_result = policy_iteration(env, gamma=1.0, theta=1e-10)
vi_result = value_iteration(env, gamma=1.0, theta=1e-10)
print("PI outer iterations:", pi_result.iterations)
print("VI sweeps:", vi_result.iterations)
print("values agree:", np.allclose(pi_result.values, vi_result.values))
print("policies agree:", np.allclose(pi_result.policy, vi_result.policy))
print("optimal value grid:\n", vi_result.values.reshape(4, 4))
print("policy labels:")
print(np.asarray(policy_labels(env, vi_result.policy)).reshape(4, 4))

## 5. 收敛阈值消融

`theta` 控制何时停止，不是学习率。阈值更严通常需要更多 sweeps；但只要动作 gap 足够大，策略可能早已相同。

In [ ]:
for theta in [1e-2, 1e-4, 1e-8, 1e-12]:
    result = iterative_policy_evaluation(
        env, random_policy, gamma=1.0, theta=theta, max_iterations=50_000
    )
    error = np.max(np.abs(result.values - evaluation.values))
    print(f"theta={theta:>7.0e}  sweeps={result.iterations:4d}  max error={error:.3e}")

## 6. 可视化最优价值与随机最优策略

多个箭头表示动作价值并列；它们都是到最近终点的最短路径。

In [ ]:
plot_value_and_policy(env, vi_result.values, vi_result.policy)
plt.show()

## 7. 思考题

<details><summary>为什么 `gamma=1` 的 GridWorld 仍有有限价值？</summary>它是 episodic stochastic-shortest-path 问题；均匀随机策略以概率 1 最终进入终点。</details>

<details><summary>为什么 terminal state 还保留四个动作？</summary>统一 `(S,A)` 数组形状；四个动作都是奖励 0、立即终止的吸收自环。</details>

<details><summary>为什么不能只比较 PI 和 VI 的 iterations？</summary>一次 PI 外层迭代包含完整策略评估的很多 sweeps；应比较总 backup 次数或时间。</details>

完整自检：`python eval/run.py`。下一任务将移除“已知完整模型”假设，改用 Monte Carlo 和 TD 样本学习。